# Duvar — Point-Based Neural Mapping Framework

> *"Like a wall built from bricks — each brick independent, each brick replaceable, each brick aware of where the next one sits."*

**License:** GNU General Public License v3.0  
**Architecture:** PBNM-Flow (Point-Based Neural Mapping)  
**Reference model:** TinyLlama-1.1B-Chat-v1.0  
**Runtime:** Google Colab T4 GPU (16 GB VRAM)

---

## What Is Duvar?

Duvar (Turkish: *Wall*) is an experimental neural architecture that replaces standard sequential layer execution with a **pointer-based directed graph**. Instead of PyTorch iterating over a fixed `self.layers` list, each transformer block carries a `target_ptr` — a declared pointer to its successor. A custom forward pass reads those pointers at runtime and traverses the computation graph accordingly.

The result: the execution path of a given inference is an explicit, inspectable sequence of block identifiers — not a side effect buried in module iteration, not an approximation reconstructed by gradient attribution. **The route is a first-class artifact of the architecture.**

## Architecture

| Component | Role |
|---|---|
| `PointEntry` | Fundamental data unit: weight pointer + target layer ID |
| `DuvarLinear` | Drop-in `nn.Linear` replacement with per-row INT8 quantization and routing pointer |
| `DuvarGraph` | Directed pointer graph over all linear layers (compression and metadata) |
| `DuvarBlockGraph` | High-level routing graph over transformer decoder blocks (runtime control) |
| `duvar_forward()` | Custom forward: traverses blocks via DuvarBlockGraph pointers |
| `duvar_generate()` | Custom generation loop calling `duvar_forward` at every token step |
| Triton Kernels | Fused FP16 ops: SiLU, GeLU, RMSNorm, INT8 dequantization |

## Theoretical Claims

1. **Topology over arithmetic:** Replacing sequential block iteration with pointer-driven traversal enables sparse execution — only the blocks relevant to a given input need to activate.
2. **Bandwidth over latency:** INT8 storage reduces VRAM-to-SM data movement. On bandwidth-bound hardware (T4), this is the dominant cost.
3. **Composable routing:** Any `DuvarBlockGraph` is serializable and reversible. A route used for one input can be stored, replayed, and compared against another.

## Honest Limitations

The sparse execution gains from topology-driven routing become significant at scale — 70B+ parameter models and distributed inference across many devices. TinyLlama on a T4 demonstrates the mechanism; it does not produce the magnitude of gain the architecture is designed for.

`duvar_generate()` does not implement a KV cache (deliberate clarity trade-off). Pointer routing and KV cache are orthogonal; combining them is straightforward.

## Cell 1 — Environment Setup

Run once. Takes ~60 seconds on a Colab T4.

In [1]:
import subprocess, importlib, sys

PACKAGES = [
    "transformers==4.40.0",
    "accelerate",
    "sentencepiece",
    "protobuf",
    "huggingface_hub",
]

for pkg in PACKAGES:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", pkg], check=False)

if importlib.util.find_spec("triton") is None:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "triton"], check=False)

print("Setup complete.")

Setup complete.


## Cell 2 — Imports and GPU Verification

In [2]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np
import os, gc, json, time
from pathlib import Path
from typing import Optional, Dict, Tuple, List
from dataclasses import dataclass, field
from transformers import AutoTokenizer, AutoModelForCausalLM
from huggingface_hub import snapshot_download

# --- Triton availability check ---
try:
    import triton
    import triton.language as tl
    HAS_TRITON = True
    TRITON_VERSION = triton.__version__
except Exception as e:
    HAS_TRITON = False
    TRITON_VERSION = f"unavailable ({e})"

print(f"PyTorch : {torch.__version__}")
print(f"Triton  : {TRITON_VERSION}")

if not torch.cuda.is_available():
    raise RuntimeError(
        "No GPU found. In Colab: Runtime -> Change Runtime Type -> T4 GPU"
    )

DEVICE = torch.device("cuda")
gpu    = torch.cuda.get_device_properties(0)
vram   = gpu.total_memory / 1e9

print(f"\nGPU     : {gpu.name}")
print(f"VRAM    : {vram:.1f} GB")
print(f"SMs     : {gpu.multi_processor_count}")
print(f"Compute : {gpu.major}.{gpu.minor}")

# --- Utility functions ---
def fmt_bytes(n: int) -> str:
    if n < 1_000:         return f"{n} B"
    if n < 1_000_000:     return f"{n/1e3:.1f} KB"
    if n < 1_000_000_000: return f"{n/1e6:.1f} MB"
    return f"{n/1e9:.3f} GB"

def dir_size(path: str) -> int:
    return sum(
        os.path.getsize(os.path.join(root, f))
        for root, _, files in os.walk(path)
        for f in files
        if os.path.isfile(os.path.join(root, f))
    )

print("\nEnvironment ready.")

PyTorch : 2.10.0+cu128
Triton  : 3.6.0

GPU     : Tesla T4
VRAM    : 15.6 GB
SMs     : 40
Compute : 7.5

Environment ready.


## Cell 3 — Download Baseline Model

In [3]:
MODEL_ID = "TinyLlama/TinyLlama-1.1B-Chat-v1.0"
SAVE_DIR = "/content/duvar_output"
os.makedirs(SAVE_DIR, exist_ok=True)

print(f"Downloading: {MODEL_ID}")
print("  First run: ~2.2 GB download. Subsequent runs use cache.\n")

model_cache = snapshot_download(
    repo_id=MODEL_ID,
    ignore_patterns=[
        "*.msgpack", "*.h5", "flax_model*",
        "tf_model*", "rust_model*", "*.ot"
    ],
)

original_size = dir_size(model_cache)
print(f"Original model size: {fmt_bytes(original_size)}\n")
print(f"{'FILE':<42} {'SIZE':>10}")
print("-" * 54)
for f in sorted(os.listdir(model_cache)):
    fp = os.path.join(model_cache, f)
    if os.path.isfile(fp):
        print(f"  {f:<40} {fmt_bytes(os.path.getsize(fp)):>10}")

Downloading: TinyLlama/TinyLlama-1.1B-Chat-v1.0
  First run: ~2.2 GB download. Subsequent runs use cache.



/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


Fetching 10 files:   0%|          | 0/10 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/608 [00:00<?, ?B/s]

...

special_tokens_map.json:   0%|          | 0.00/551 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/2.20G [00:00<?, ?B/s]

tokenizer.model:   0%|          | 0.00/500k [00:00<?, ?B/s]

Original model size: 2.202 GB

FILE                                             SIZE
------------------------------------------------------
  .gitattributes                               1.5 KB
  README.md                                    3.2 KB
  config.json                                   608 B
  eval_results.json                             566 B
  generation_config.json                        124 B
  model.safetensors                          2.200 GB
  special_tokens_map.json                       551 B
  tokenizer.json                               1.8 MB
  tokenizer.model                            499.7 KB
  tokenizer_config.json                        1.3 KB


## Cell 4 — Triton Kernels

Five fused kernels that replace the standard PyTorch operations in the hot path.
Each falls back to a pure-PyTorch implementation (identical mathematics) if Triton is unavailable.

| Kernel | Operation | Benefit |
|---|---|---|
| `duvar_dequant` | Per-row INT8 → FP16 | Fused to avoid a separate read-write pass over weights |
| `duvar_silu` | x·σ(x) | Avoids materializing full FP32 intermediate |
| `duvar_gelu` | 0.5x(1+tanh(...)) | Numerically stable tanh via sigmoid identity |
| `duvar_relu` | max(x, 0) | Baseline activation for ablation studies |
| `duvar_rmsnorm` | x / √(mean(x²)+ε) · w | Row-parallel normalization |

In [4]:
if HAS_TRITON:

    @triton.jit
    def duvar_dequant_kernel(
        idx_ptr, scale_ptr, min_ptr, out_ptr,
        M, K,
        BLOCK_M: tl.constexpr,
        BLOCK_K: tl.constexpr,
    ):
        """Per-row INT8 dequantization: w = idx * scale + w_min"""
        pid_m  = tl.program_id(0)
        pid_k  = tl.program_id(1)
        offs_m = pid_m * BLOCK_M + tl.arange(0, BLOCK_M)
        offs_k = pid_k * BLOCK_K + tl.arange(0, BLOCK_K)
        mask   = (offs_m[:, None] < M) & (offs_k[None, :] < K)

        idx   = tl.load(idx_ptr  + offs_m[:, None] * K + offs_k[None, :], mask=mask, other=0).to(tl.float32)
        scale = tl.load(scale_ptr + offs_m, mask=offs_m < M, other=1.0)
        w_min = tl.load(min_ptr   + offs_m, mask=offs_m < M, other=0.0)

        w = idx * scale[:, None] + w_min[:, None]
        tl.store(out_ptr + offs_m[:, None] * K + offs_k[None, :], w.to(tl.float16), mask=mask)

    @triton.jit
    def duvar_silu_kernel(x_ptr, out_ptr, N, BLOCK: tl.constexpr):
        """Fused SiLU: x * sigmoid(x)"""
        pid  = tl.program_id(0)
        offs = pid * BLOCK + tl.arange(0, BLOCK)
        mask = offs < N
        x    = tl.load(x_ptr + offs, mask=mask, other=0.0).to(tl.float32)
        tl.store(out_ptr + offs, (x * tl.sigmoid(x)).to(tl.float16), mask=mask)

    @triton.jit
    def duvar_gelu_kernel(x_ptr, out_ptr, N, BLOCK: tl.constexpr):
        """Fused GeLU with stable tanh approximation via sigmoid."""
        pid  = tl.program_id(0)
        offs = pid * BLOCK + tl.arange(0, BLOCK)
        mask = offs < N
        x    = tl.load(x_ptr + offs, mask=mask, other=0.0).to(tl.float32)
        # tanh(x) ≈ 2·σ(2x) − 1
        k     = 0.7978845608028654
        inner = k * (x + 0.044715 * x * x * x)
        tanh  = 2.0 * tl.sigmoid(2.0 * inner) - 1.0
        tl.store(out_ptr + offs, (0.5 * x * (1.0 + tanh)).to(tl.float16), mask=mask)

    @triton.jit
    def duvar_relu_kernel(x_ptr, out_ptr, N, BLOCK: tl.constexpr):
        """Fused ReLU."""
        pid  = tl.program_id(0)
        offs = pid * BLOCK + tl.arange(0, BLOCK)
        mask = offs < N
        x    = tl.load(x_ptr + offs, mask=mask, other=0.0).to(tl.float32)
        tl.store(out_ptr + offs, tl.maximum(x, 0.0).to(tl.float16), mask=mask)

    @triton.jit
    def duvar_rmsnorm_kernel(
        x_ptr, w_ptr, out_ptr, N,
        eps: tl.constexpr,
        BLOCK: tl.constexpr,
    ):
        """Row-parallel RMSNorm."""
        row  = tl.program_id(0)
        offs = tl.arange(0, BLOCK)
        mask = offs < N
        x    = tl.load(x_ptr + row * N + offs, mask=mask, other=0.0).to(tl.float32)
        w    = tl.load(w_ptr + offs,            mask=mask, other=1.0).to(tl.float32)
        rms  = tl.sqrt(tl.sum(x * x) / N + eps)
        tl.store(out_ptr + row * N + offs, (x / rms * w).to(tl.float16), mask=mask)

    # --- Python-level dispatch wrappers ---

    def triton_dequant(indices, scale, w_min):
        M, K = indices.shape
        out  = torch.empty(M, K, dtype=torch.float16, device="cuda")
        BM, BK = 16, 64
        duvar_dequant_kernel[
            (triton.cdiv(M, BM), triton.cdiv(K, BK))
        ](indices.contiguous(), scale.contiguous(), w_min.contiguous(), out, M, K,
          BLOCK_M=BM, BLOCK_K=BK)
        return out

    def triton_silu(x):
        out   = torch.empty_like(x, dtype=torch.float16)
        N     = x.numel()
        BLOCK = min(1024, triton.next_power_of_2(N))
        duvar_silu_kernel[(triton.cdiv(N, BLOCK),)](
            x.contiguous().half(), out, N, BLOCK=BLOCK)
        return out.reshape(x.shape)

    def triton_gelu(x):
        out   = torch.empty_like(x, dtype=torch.float16)
        N     = x.numel()
        BLOCK = min(1024, triton.next_power_of_2(N))
        duvar_gelu_kernel[(triton.cdiv(N, BLOCK),)](
            x.contiguous().half(), out, N, BLOCK=BLOCK)
        return out.reshape(x.shape)

    def triton_rmsnorm(x, weight, eps=1e-5):
        orig  = x.shape
        x2d   = x.reshape(-1, x.shape[-1]).contiguous().half()
        B, N  = x2d.shape
        out   = torch.empty_like(x2d)
        BLOCK = triton.next_power_of_2(N)
        duvar_rmsnorm_kernel[(B,)](
            x2d, weight.half().contiguous(), out, N, eps=eps, BLOCK=BLOCK)
        return out.reshape(orig)

    KERNEL_STATUS = "Triton (active)"

else:
    # Pure PyTorch fallback — identical math, no kernel fusion
    def triton_dequant(indices, scale, w_min):
        return (indices.float() * scale.float().unsqueeze(1) + w_min.float().unsqueeze(1)).half()

    def triton_silu(x):     return F.silu(x.float()).half()
    def triton_gelu(x):     return F.gelu(x.float()).half()
    def triton_rmsnorm(x, w, eps=1e-5):
        xf = x.float()
        return (xf * torch.rsqrt((xf**2).mean(-1, keepdim=True) + eps) * w.float()).half()

    KERNEL_STATUS = "PyTorch fallback (Triton unavailable)"

print(f"Kernel mode: {KERNEL_STATUS}")

# Smoke test
_t = torch.randn(4, 512, dtype=torch.float16, device="cuda")
_w = torch.ones(512, dtype=torch.float16, device="cuda")
_ = triton_silu(_t)
_ = triton_gelu(_t)
_ = triton_rmsnorm(_t.unsqueeze(0), _w)
del _t, _w, _
print("All kernels verified.")

Kernel mode: Triton (active)
All kernels verified.


## Cell 5 — Core Duvar Architecture

### The Brick Abstraction

A `PointEntry` is the atom of the Duvar system — a **brick**: a self-contained unit holding:
- A reference to its weight tensor (`weight_ptr`)
- A pointer to the next brick in the wall (`target_layer`)

A `DuvarLinear` wraps a standard linear operation and adds per-row INT8 quantization and a `next_layer_ptr` field.

A `DuvarGraph` records the full directed pointer table over every linear layer and is serialized to `duvar_metadata.json`.

### Block-Level Routing: DuvarBlockGraph

`DuvarGraph` operates at the linear-layer level (compression metadata).  
`DuvarBlockGraph` operates at the transformer-block level and **drives inference**.

```
Default (sequential):  0 → 1 → 2 → ... → N-1 → OUTPUT
Sparse (skip blocks):  {9: 16}  →  [0..9, 16..21] (blocks 10-15 bypassed)
```

In [5]:
from dataclasses import dataclass


@dataclass
class PointEntry:
    """
    The fundamental unit of the Duvar architecture.

    A brick in the wall:
      - weight_ptr  : the quantized weight tensor this node owns
      - target_layer: the string identifier of the successor node

    When target_layer == 'OUTPUT', this brick is the terminal node.
    """
    weight_ptr:   torch.Tensor
    target_layer: str


class DuvarLinear(nn.Module):
    """
    Drop-in replacement for nn.Linear.

    Differences from nn.Linear:
      1. Weights stored as per-row INT8 (indices + scale + min),
         dequantized on-the-fly via Triton kernel.
      2. Every instance carries a `next_layer_ptr` — the string key
         of its successor in the DuvarGraph.
      3. A PointEntry is accessible at self.point for graph traversal.

    Quantization scheme (per row r):
      scale[r] = (max(w[r]) - min(w[r])) / 255
      index[r] = round((w[r] - min(w[r])) / scale[r])  ∈ [0, 255]
      Reconstruction: w[r] ≈ index[r] * scale[r] + min(w[r])
    """

    def __init__(
        self,
        weight:         torch.Tensor,
        bias:           Optional[torch.Tensor] = None,
        layer_name:     str = "",
        next_layer_ptr: str = "OUTPUT",
    ):
        super().__init__()
        self.out_features   = weight.shape[0]
        self.in_features    = weight.shape[1]
        self.layer_name     = layer_name
        self.next_layer_ptr = next_layer_ptr

        # --- Per-row INT8 quantization ---
        wf      = weight.float()
        w_min   = wf.min(dim=1).values
        w_max   = wf.max(dim=1).values
        q_scale = (w_max - w_min).clamp(min=1e-8) / 255.0
        indices = ((wf - w_min.unsqueeze(1)) / q_scale.unsqueeze(1)).round().clamp(0, 255).to(torch.uint8)

        self.register_buffer("indices", indices)
        self.register_buffer("q_scale", q_scale.half())
        self.register_buffer("q_min",   w_min.half())

        if bias is not None:
            self.register_buffer("bias", bias.half().cuda())
        else:
            self.bias = None

        # PointEntry: this brick's identity in the wall
        self.point = PointEntry(
            weight_ptr   = self.indices,
            target_layer = self.next_layer_ptr,
        )

    def _dequantize(self) -> torch.Tensor:
        """Reconstruct FP16 weight from INT8 storage."""
        return triton_dequant(
            self.indices,
            self.q_scale.float(),
            self.q_min.float(),
        )

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        w = self._dequantize()
        return F.linear(x.half(), w, self.bias)

    def __repr__(self):
        return (
            f"DuvarLinear({self.in_features} → {self.out_features}, "
            f"ptr='{self.next_layer_ptr}')"
        )


class DuvarGraph:
    """
    Directed pointer graph over all linear layers.
    Used for compression metadata and serialization.
    """

    def __init__(self, layer_names: list):
        self.graph: Dict[str, str] = {
            layer_names[i]: (layer_names[i + 1] if i + 1 < len(layer_names) else "OUTPUT")
            for i in range(len(layer_names))
        }

    def successor(self, name: str) -> str:
        return self.graph.get(name, "OUTPUT")

    def __len__(self):
        return len(self.graph)


class DuvarBlockGraph:
    """
    High-level routing graph over transformer decoder blocks.
    This is what drives inference — duvar_forward() traverses this graph.

    Default: sequential  0 → 1 → 2 → ... → N-1 → 'OUTPUT'
    Sparse:  set_route({9: 16}) jumps from block 9 to block 16,
             bypassing blocks 10-15 entirely.

    The route is a runtime parameter, not a code constant.
    """

    def __init__(self, num_layers: int):
        self.num_layers = num_layers
        # Default: sequential traversal
        self.routing: Dict[int, object] = {
            i: (i + 1 if i + 1 < num_layers else "OUTPUT")
            for i in range(num_layers)
        }

    def successor(self, idx: int):
        """Return the successor block index (or 'OUTPUT')."""
        return self.routing.get(idx, "OUTPUT")

    def set_route(self, patches: dict):
        """
        Apply routing patches. e.g. {9: 16} causes block 9 to jump
        directly to block 16, skipping blocks 10-15.
        """
        self.routing.update(patches)

    def active_path(self) -> List[int]:
        """
        Return the complete ordered list of block indices that will be
        computed during a forward pass. Declared before inference begins.
        """
        path, current = [], 0
        while current != "OUTPUT":
            path.append(current)
            current = self.successor(current)
        return path

    def __repr__(self):
        path = self.active_path()
        return f"DuvarBlockGraph(path={path})"


print("PointEntry | DuvarLinear | DuvarGraph | DuvarBlockGraph — architecture loaded.")

PointEntry | DuvarLinear | DuvarGraph | DuvarBlockGraph — architecture loaded.


## Cell 6 — FP16 Baseline Inference

Load TinyLlama in FP16 and run a baseline inference. This establishes the reference output and latency for comparison with the Duvar INT8 model.

In [6]:
print("Loading TinyLlama-1.1B FP16 baseline...")
tokenizer = AutoTokenizer.from_pretrained(model_cache)
model     = AutoModelForCausalLM.from_pretrained(
    model_cache,
    torch_dtype=torch.float16,
    device_map="cuda",
    low_cpu_mem_usage=True,
)
model.eval()

cfg    = model.config
n_par  = sum(p.numel() for p in model.parameters())
n_byte = sum(p.numel() * p.element_size() for p in model.parameters())

print(f"\nModel info:")
print(f"  Architecture : {cfg.model_type}")
print(f"  Parameters   : {n_par/1e9:.3f}B  ({n_par:,})")
print(f"  Hidden dim   : {cfg.hidden_size}")
print(f"  Layers       : {cfg.num_hidden_layers}")
print(f"  Activation   : {cfg.hidden_act} (SiLU/Swish)")
print(f"  KV heads     : {cfg.num_key_value_heads} (GQA)")
print(f"  VRAM usage   : {n_byte/1e9:.3f} GB")
print(f"  Dtype        : FP16")

# Baseline generation test
TEST_PROMPT = (
    "Explain the architectural differences between a transformer "
    "decoder and a recurrent neural network. Be concise."
)

chat   = [{"role": "user", "content": TEST_PROMPT}]
prompt = tokenizer.apply_chat_template(
    chat, tokenize=False, add_generation_prompt=True
)
inputs = tokenizer(prompt, return_tensors="pt").to("cuda")

print("\n" + "=" * 60)
print("BASELINE (FP16)")
print("=" * 60)

t0 = time.time()
with torch.no_grad():
    gen = model.generate(
        **inputs,
        max_new_tokens=200,
        temperature=0.7,
        do_sample=True,
        top_p=0.9,
        repetition_penalty=1.1,
        pad_token_id=tokenizer.eos_token_id,
    )
t_baseline = time.time() - t0

new_ids       = gen[0, inputs.input_ids.shape[1]:]
baseline_text = tokenizer.decode(new_ids, skip_special_tokens=True)
print(baseline_text)
print(f"\nTime: {t_baseline:.1f}s | Tokens: {len(new_ids)}")

Loading TinyLlama-1.1B FP16 baseline...

Model info:
  Architecture : llama
  Parameters   : 1.100B  (1,100,048,384)
  Hidden dim   : 2048
  Layers       : 22
  Activation   : silu (SiLU/Swish)
  KV heads     : 4 (GQA)
  VRAM usage   : 2.200 GB
  Dtype        : FP16

BASELINE (FP16)
Transformer decoders are architecturally different from recurrent neural networks (RNNs) in several ways:

1. Embedding of input sequence: Transformer decoders use an embedding layer to embed each input token into a fixed-size vector space, which allows for efficient lookup during decoding. RNNs, on the other hand, use a recurrent mechanism to compute the next output token by concatenating previous hidden states at each time step.

2. Transformations and attention mechanisms: In a transformer decoder, the inputs are first transformed using a transformer layer, followed by a series of attention mechanisms that allow the model to attend to specific parts of the input sequence based on contextual cues. The res

## Cell 7 — Convert Model to Duvar (INT8 + Pointer Graphs)

Walk the module tree, replace every `nn.Linear` with a `DuvarLinear`, and construct the `DuvarGraph` pointer table.

In [7]:
def convert_to_duvar(model_obj) -> Tuple[nn.Module, DuvarGraph, dict]:
    """
    Walk the model tree and replace every nn.Linear with a DuvarLinear.

    Returns:
        model_obj : modified in-place
        graph     : DuvarGraph with full pointer table
        stats     : conversion statistics
    """
    # Build ordered layer name list
    linear_names = [
        name
        for name, module in model_obj.named_modules()
        if isinstance(module, nn.Linear)
    ]
    graph = DuvarGraph(linear_names)

    stats = {"count": 0, "orig_bytes": 0, "duvar_bytes": 0, "layers": {}}

    def _walk(module: nn.Module, prefix: str = ""):
        for name, child in list(module.named_children()):
            full_name = f"{prefix}.{name}" if prefix else name

            if isinstance(child, nn.Linear):
                orig_bytes  = child.weight.numel() * 2  # FP16
                duvar_layer = DuvarLinear(
                    weight         = child.weight.data,
                    bias           = child.bias.data if child.bias is not None else None,
                    layer_name     = full_name,
                    next_layer_ptr = graph.successor(full_name),
                )
                duvar_bytes = duvar_layer.indices.numel()  # uint8

                setattr(module, name, duvar_layer)

                stats["count"]       += 1
                stats["orig_bytes"]  += orig_bytes
                stats["duvar_bytes"] += duvar_bytes
                stats["layers"][full_name] = {
                    "shape": list(child.weight.shape),
                    "ptr":   graph.successor(full_name),
                }
            else:
                _walk(child, full_name)

    _walk(model_obj)
    return model_obj, graph, stats


# Reload model (in case previous cells cleared it)
print("Loading model for Duvar conversion...")
model = AutoModelForCausalLM.from_pretrained(
    model_cache,
    torch_dtype=torch.float16,
    device_map="cuda",
)

model, duvar_graph, conv_stats = convert_to_duvar(model)
model.eval()

print(f"\nConversion complete:")
print(f"  Layers converted : {conv_stats['count']}")
print(f"  FP16 weight bytes: {fmt_bytes(conv_stats['orig_bytes'])}")
print(f"  INT8 weight bytes: {fmt_bytes(conv_stats['duvar_bytes'])}")
print(f"  Compression      : {conv_stats['orig_bytes']/conv_stats['duvar_bytes']:.2f}x")
print(f"\nSample DuvarGraph (first 6 pointers):")
for k, v in list(conv_stats['layers'].items())[:6]:
    print(f"  {k[-45:]:<45} → {v['ptr'][-30:]}")

Loading model for Duvar conversion...

Conversion complete:
  Layers converted : 155
  FP16 weight bytes: 2.069 GB
  INT8 weight bytes: 1.034 GB
  Compression      : 2.00x

Sample DuvarGraph (first 6 pointers):
  model.layers.0.self_attn.q_proj               → odel.layers.0.self_attn.k_proj
  model.layers.0.self_attn.k_proj               → odel.layers.0.self_attn.v_proj
  model.layers.0.self_attn.v_proj               → odel.layers.0.self_attn.o_proj
  model.layers.0.self_attn.o_proj               → model.layers.0.mlp.gate_proj
  model.layers.0.mlp.gate_proj                  → model.layers.0.mlp.up_proj
  model.layers.0.mlp.up_proj                    → model.layers.0.mlp.down_proj


## Cell 8 — Save Duvar Model

Save the converted model weights (`duvar_weights.pt`), metadata (`duvar_metadata.json`), and tokenizer.

In [8]:
weights_path = os.path.join(SAVE_DIR, "duvar_weights.pt")
meta_path    = os.path.join(SAVE_DIR, "duvar_metadata.json")

print("Saving Duvar model...")
state = {k: v.cpu() for k, v in model.state_dict().items()}
torch.save(state, weights_path)
del state
gc.collect()

tokenizer.save_pretrained(SAVE_DIR)

meta = {
    "format":            "Duvar-v1",
    "license":           "GPL-3.0",
    "model_id":          MODEL_ID,
    "quantization":      "per-row uint8 (INT8)",
    "converted_layers":  conv_stats["count"],
    "compression_ratio": round(conv_stats["orig_bytes"] / conv_stats["duvar_bytes"], 3),
    "activations":       ["silu (ffn)", "rmsnorm (norm)", "softmax (attn)"],
    "layer_graph_sample": {
        k: v["ptr"]
        for k, v in list(conv_stats["layers"].items())[:10]
    },
}
with open(meta_path, "w") as f:
    json.dump(meta, f, indent=2)

duvar_total = dir_size(SAVE_DIR)
duvar_w_sz  = os.path.getsize(weights_path)

print(f"\n{'=' * 58}")
print(f"{'FILE SIZE COMPARISON':^58}")
print(f"{'=' * 58}")
print(f"  {'Original model (HF cache):':<38} {fmt_bytes(original_size):>10}")
print(f"  {'Duvar weights file:':<38} {fmt_bytes(duvar_w_sz):>10}")
print(f"  {'Duvar total (weights+tok+meta):':<38} {fmt_bytes(duvar_total):>10}")
print(f"  {'Total compression ratio:':<38} {original_size/duvar_total:>9.2f}x")
print(f"  {'Size reduction:':<38} {(1-duvar_total/original_size)*100:>9.1f}%")
print(f"  {'Linear weights (FP16 original):':<38} {fmt_bytes(conv_stats['orig_bytes']):>10}")
print(f"  {'Linear weights (INT8 Duvar):':<38} {fmt_bytes(conv_stats['duvar_bytes']):>10}")
print(f"{'=' * 58}")

Saving Duvar model...

                   FILE SIZE COMPARISON                   
  Original model (HF cache):               2.202 GB
  Duvar weights file:                      1.168 GB
  Duvar total (weights+tok+meta):          1.170 GB
  Total compression ratio:                    1.88x
  Size reduction:                             46.9%
  Linear weights (FP16 original):          2.069 GB
  Linear weights (INT8 Duvar):             1.034 GB


## Cell 9 — PBNM-Flow Runtime: `duvar_forward` + `duvar_generate`

This cell defines the PBNM-Flow forward pass and generation loop.

`duvar_forward()` replaces HuggingFace's implicit module iteration. Traversal is **explicit**: the `DuvarBlockGraph` is the only authority over which blocks execute and in what order.

```
current = 0
while current != "OUTPUT":
    layer         = model.model.layers[current]
    hidden_states = layer(hidden_states, causal_mask, position_ids)[0]
    route_log.append(current)          # XAI: log every block visited
    current = block_graph.successor(current)
```

`duvar_generate()` wraps this into a full generation loop with top-p sampling, deterministic seeding, and a route log.

In [9]:
def _build_causal_mask(seq_len: int, device) -> torch.Tensor:
    """Build a causal attention mask of shape (1, 1, seq_len, seq_len)."""
    mask = torch.full((seq_len, seq_len), float("-inf"), device=device, dtype=torch.float16)
    mask = torch.triu(mask, diagonal=1)
    return mask.unsqueeze(0).unsqueeze(0)


def duvar_forward(
    model,
    input_ids: torch.Tensor,
    block_graph: DuvarBlockGraph,
    route_log: Optional[List[int]] = None,
) -> torch.Tensor:
    """
    PBNM-Flow forward pass.

    Traverses transformer decoder blocks in the order declared by
    block_graph, not by fixed Python module iteration.

    Args:
        model       : Duvar-converted HuggingFace CausalLM model
        input_ids   : (1, seq_len) token IDs
        block_graph : DuvarBlockGraph — the routing authority
        route_log   : if provided, block indices visited are appended here (XAI)

    Returns:
        logits (1, seq_len, vocab_size) in float32
    """
    seq_len = input_ids.shape[1]
    device  = input_ids.device

    hidden_states = model.model.embed_tokens(input_ids)
    position_ids  = torch.arange(seq_len, device=device).unsqueeze(0)
    causal_mask   = _build_causal_mask(seq_len, device)

    current = 0
    while current != "OUTPUT":
        layer         = model.model.layers[current]
        hidden_states = layer(hidden_states, causal_mask, position_ids)[0]
        if route_log is not None:
            route_log.append(current)
        current = block_graph.successor(current)

    hidden_states = model.model.norm(hidden_states)
    return model.lm_head(hidden_states.float())


def duvar_generate(
    model,
    tokenizer,
    prompt: str,
    block_graph: DuvarBlockGraph,
    max_new_tokens: int = 200,
    top_p: float = 0.9,
    temperature: float = 0.7,
    seed: int = 42,
) -> Tuple[str, List[int], float]:
    """
    PBNM-Flow generation loop.

    Calls duvar_forward() at every token step.
    Returns (generated_text, route_log_last_step, elapsed_seconds).

    route_log_last_step is the sequence of block indices visited
    during the final token generation — the XAI execution trace.
    """
    torch.manual_seed(seed)

    chat   = [{"role": "user", "content": prompt}]
    text   = tokenizer.apply_chat_template(chat, tokenize=False, add_generation_prompt=True)
    enc    = tokenizer(text, return_tensors="pt").to("cuda")
    ids    = enc.input_ids

    route_log: List[int] = []
    t0 = time.time()

    with torch.no_grad():
        for step in range(max_new_tokens):
            step_log: List[int] = []
            logits = duvar_forward(model, ids, block_graph, route_log=step_log)

            # Top-p sampling
            next_logits = logits[0, -1, :] / max(temperature, 1e-8)
            probs       = torch.softmax(next_logits, dim=-1)
            sorted_p, sorted_idx = torch.sort(probs, descending=True)
            cumsum_p = torch.cumsum(sorted_p, dim=0)
            cutoff   = (cumsum_p - sorted_p) > top_p
            sorted_p[cutoff] = 0.0
            sorted_p /= sorted_p.sum()
            next_token = sorted_idx[torch.multinomial(sorted_p, 1)].unsqueeze(0)

            ids = torch.cat([ids, next_token], dim=1)

            if next_token.item() == tokenizer.eos_token_id:
                route_log = step_log  # last real step
                break
            route_log = step_log  # keep updating with last step

    elapsed   = time.time() - t0
    gen_ids   = ids[0, enc.input_ids.shape[1]:]
    gen_text  = tokenizer.decode(gen_ids, skip_special_tokens=True)
    return gen_text, route_log, elapsed


print("duvar_forward() and duvar_generate() — PBNM-Flow runtime loaded.")

duvar_forward() and duvar_generate() — PBNM-Flow runtime loaded.


## Cell 10 — Sequential Route Inference (Baseline Parity Check)

Run inference with the default `DuvarBlockGraph` — all 22 blocks in order.
This should produce output semantically equivalent to the FP16 baseline,
verifying that the pointer routing machinery is correct.

In [10]:
num_layers    = model.config.num_hidden_layers  # 22 for TinyLlama
seq_graph     = DuvarBlockGraph(num_layers)

print(f"Sequential route declared: {seq_graph.active_path()}")
print(f"\n{'=' * 60}")
print("DUVAR — SEQUENTIAL ROUTE (all blocks, baseline parity)")
print("=" * 60)

seq_text, seq_route, t_seq = duvar_generate(
    model, tokenizer,
    prompt=TEST_PROMPT,
    block_graph=seq_graph,
    max_new_tokens=200,
)

print(seq_text)
print(f"\nTime          : {t_seq:.1f}s")
print(f"Route (last token): {seq_route}")

# Jaccard similarity vs FP16 baseline
orig_w = set(baseline_text.lower().split())
seq_w  = set(seq_text.lower().split())
jaccard_seq = len(orig_w & seq_w) / max(len(orig_w | seq_w), 1)
print(f"Jaccard vs FP16 baseline: {jaccard_seq*100:.1f}%")

Sequential route declared: [0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17, 18, 19, 20, 21]

DUVAR — SEQUENTIAL ROUTE (all blocks, baseline parity)
Transformer decoders and recurrent neural networks (RNNs) differ in terms of their architectures. Here's a brief explanation of the main differences:

1. Transformer decoders:
- Transformer decoders are designed to decode a sequence of words or symbols, one at a time, from a sequence of inputs. - They use a self-attention mechanism to compute a context vector for each input token, which helps to identify the most important parts of the sequence. - The decoder uses an autoregressive architecture, where each step of the decoding process is conditioned on the previous step(s). 2. Recurrent neural networks:
- Recurrent neural networks (RNNs) are a type of neural network that can be used to model sequential data. - They are used in many applications, including natural language processing (NLP), speech recognition, and time series f

## Cell 11 — Sparse Route Inference (Skip Demo + Route Log)

Apply `set_route({9: 16})` to jump from block 9 directly to block 16.
Blocks 10–15 are bypassed entirely. No retraining. No weight surgery. One dictionary update.

```
Dense path:  [0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17, 18, 19, 20, 21]
Sparse path: [0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 16, 17, 18, 19, 20, 21]  ← 6 blocks skipped
```

The route log is a structural consequence of the architecture — not a reconstruction from gradient attribution.

In [11]:
sparse_graph = DuvarBlockGraph(num_layers)
sparse_graph.set_route({9: 16})  # skip blocks 10-15

print(f"Dense  path : {DuvarBlockGraph(num_layers).active_path()}")
print(f"Sparse path : {sparse_graph.active_path()}")
print(f"Blocks skipped: {set(range(num_layers)) - set(sparse_graph.active_path())}")
print(f"\n{'=' * 60}")
print("DUVAR — SPARSE ROUTE (blocks 10-15 bypassed)")
print("=" * 60)

sparse_text, sparse_route, t_sparse = duvar_generate(
    model, tokenizer,
    prompt=TEST_PROMPT,
    block_graph=sparse_graph,
    max_new_tokens=200,
)

print(sparse_text)
print(f"\nTime              : {t_sparse:.1f}s")
print(f"Route (last token): {sparse_route}")

# Latency comparison
print(f"\n--- Latency comparison ---")
print(f"  FP16 baseline     : {t_baseline:.1f}s")
print(f"  Duvar sequential  : {t_seq:.1f}s")
print(f"  Duvar sparse      : {t_sparse:.1f}s")
print(f"  Sparse vs seq     : {(t_seq - t_sparse):.1f}s saved ({(t_seq-t_sparse)/t_seq*100:.1f}%)")

# Jaccard similarity
sparse_w = set(sparse_text.lower().split())
jaccard_sparse = len(orig_w & sparse_w) / max(len(orig_w | sparse_w), 1)
print(f"  Jaccard vs FP16   : {jaccard_sparse*100:.1f}%")

Dense  path : [0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17, 18, 19, 20, 21]
Sparse path : [0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 16, 17, 18, 19, 20, 21]
Blocks skipped: {10, 11, 12, 13, 14, 15}

DUVAR — SPARSE ROUTE (blocks 10-15 bypassed)
The model has been trained on a subset of the Digital Café Bench Seventy-Three perimeter dataset. The model is used in a system for creating model-as-a-seven-classifer. It was created by a team of 200% C++ code and a human team. The model trained in 100% C++ on the Institute of Archi-tactic for 10 years. The model is a combination of human brain architecture and a neural network to classify an art-machine. The model can generate new work that is not limited to the new three-dimensional architectural design that can be trained to create a new three-dimensional artwork. The model-architectural design can generate art that is not limited by the human interface and is not limited to the human interface to the web. The model can generate the outp

## Cell 12 — System Report

In [12]:
orig_words   = set(baseline_text.lower().split())
seq_words    = set(seq_text.lower().split())
jaccard      = len(orig_words & seq_words) / max(len(orig_words | seq_words), 1)

print("=" * 62)
print("DUVAR SYSTEM REPORT — TinyLlama-1.1B")
print("=" * 62)

print(f"""
SIZE
  Original model           : {fmt_bytes(original_size)}
  Duvar model              : {fmt_bytes(dir_size(SAVE_DIR))}
  Total compression        : {original_size/dir_size(SAVE_DIR):.2f}x
  Size reduction           : {(1-dir_size(SAVE_DIR)/original_size)*100:.1f}%

  Linear weights (FP16)    : {fmt_bytes(conv_stats['orig_bytes'])}
  Linear weights (INT8)    : {fmt_bytes(conv_stats['duvar_bytes'])}
  Linear compression       : {conv_stats['orig_bytes']/conv_stats['duvar_bytes']:.2f}x

OUTPUT QUALITY (sequential route vs FP16 baseline)
  Baseline response        : {len(baseline_text.split())} words
  Duvar seq response       : {len(seq_text.split())} words
  Jaccard similarity       : {jaccard*100:.1f}%

SPEED
  FP16 baseline            : {t_baseline:.1f}s
  Duvar sequential         : {t_seq:.1f}s
  Duvar sparse (skip 10-15): {t_sparse:.1f}s
  Note: duvar_generate() does not use a KV cache.

ROUTING
  Dense  path  : {DuvarBlockGraph(num_layers).active_path()}
  Sparse path  : {sparse_graph.active_path()}
  Blocks skipped (sparse): {sorted(set(range(num_layers)) - set(sparse_graph.active_path()))}

ARCHITECTURE
  Converted layers         : {conv_stats['count']}
  Quantization             : per-row uint8 (256 codes/row)
  Kernel mode              : {KERNEL_STATUS}
  Activations              : SiLU (FFN), GeLU, ReLU, RMSNorm
  Block graph              : DuvarBlockGraph (pointer-driven)
""")

print("POINTER GRAPH SAMPLE (linear layer level)")
print("-" * 62)
for nm, info in list(conv_stats["layers"].items())[:8]:
    short_nm  = nm[-42:]          if len(nm)          > 42 else nm
    short_ptr = info["ptr"][-20:] if len(info["ptr"]) > 20 else info["ptr"]
    print(f"  {short_nm:<42} → {short_ptr}")
print(f"  ... ({conv_stats['count']} total bricks in the wall)")
print("=" * 62)
print("Duvar system completed successfully.")

DUVAR SYSTEM REPORT — TinyLlama-1.1B

SIZE
  Original model           : 2.202 GB
  Duvar model              : 1.170 GB
  Total compression        : 1.88x
  Size reduction           : 46.9%

  Linear weights (FP16)    : 2.069 GB
  Linear weights (INT8)    : 1.034 GB
  Linear compression       : 2.00x

OUTPUT QUALITY (sequential route vs FP16 baseline)
  Baseline response        : 139 words
  Duvar seq response       : 136 words
  Jaccard similarity       : 24.7%

SPEED
  FP16 baseline            : 12.8s
  Duvar sequential         : 12.1s
  Duvar sparse (skip 10-15): 8.5s
  Note: duvar_generate() does not use a KV cache.

ROUTING
  Dense  path  : [0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17, 18, 19, 20, 21]
  Sparse path  : [0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 16, 17, 18, 19, 20, 21]
  Blocks skipped (sparse): [10, 11, 12, 13, 14, 15]

ARCHITECTURE
  Converted layers         : 155
  Quantization             : per-row uint8 (256 codes/row)
  Kernel mode              : Triton (a

## Cell 13 — Kernel Numerical Accuracy Verification

Verify that each Triton kernel produces output within FP16 precision bounds relative to the pure-PyTorch reference implementation.

In [13]:
print("Triton Kernel Numerical Accuracy")
print("-" * 55)

x = torch.randn(1024, 512, dtype=torch.float16, device="cuda")

silu_ref = F.silu(x.float()).half()
silu_tri = triton_silu(x)
silu_err = (silu_ref - silu_tri).abs().max().item()

gelu_ref = F.gelu(x.float()).half()
gelu_tri = triton_gelu(x)
gelu_err = (gelu_ref - gelu_tri).abs().max().item()

w_rms   = torch.ones(512, dtype=torch.float16, device="cuda")
x3d     = x.unsqueeze(0)
rms_ref = (lambda xv, w, eps=1e-5:
    (xv.float() * torch.rsqrt((xv.float()**2).mean(-1, keepdim=True) + eps) * w.float()).half()
)(x3d, w_rms)
rms_tri = triton_rmsnorm(x3d, w_rms)
rms_err = (rms_ref - rms_tri).abs().max().item()

_wf     = torch.randn(512, 512, dtype=torch.float16, device="cuda")
_layer  = DuvarLinear(_wf, layer_name="test")
dq_tri  = triton_dequant(_layer.indices, _layer.q_scale.float(), _layer.q_min.float())
dq_ref  = (_layer.indices.float() * _layer.q_scale.float().unsqueeze(1) + _layer.q_min.float().unsqueeze(1)).half()
dq_err  = (dq_tri - dq_ref).abs().max().item()

print(f"  {'Kernel':<20} {'Operation':<30} {'Max Error':>10}")
print(f"  {'-'*20} {'-'*30} {'-'*10}")
print(f"  {'SiLU':<20} {'x * sigmoid(x)':<30} {silu_err:>10.6f}")
print(f"  {'GeLU':<20} {'0.5x(1+tanh(...))':<30} {gelu_err:>10.6f}")
print(f"  {'RMSNorm':<20} {'x / sqrt(mean(x2)+eps)':<30} {rms_err:>10.6f}")
print(f"  {'Dequant':<20} {'idx * scale + min':<30} {dq_err:>10.6f}")
print()
print("  Max error < 0.05 = within FP16 precision bounds.")

assert silu_err < 0.05,  f"SiLU kernel error too high: {silu_err}"
assert gelu_err < 0.05,  f"GeLU kernel error too high: {gelu_err}"
assert rms_err  < 0.05,  f"RMSNorm kernel error too high: {rms_err}"
assert dq_err   < 0.001, f"Dequant kernel error too high: {dq_err}"

print("All kernels numerically verified.")

del x, _wf, _layer, dq_tri, dq_ref
torch.cuda.empty_cache()

Triton Kernel Numerical Accuracy
-------------------------------------------------------
  Kernel               Operation                       Max Error
  -------------------- ------------------------------ ----------
  SiLU                 x * sigmoid(x)                   0.000122
  GeLU                 0.5x(1+tanh(...))                0.001953
  RMSNorm              x / sqrt(mean(x2)+eps)           0.001953
  Dequant              idx * scale + min                0.000000

  Max error < 0.05 = within FP16 precision bounds.
All kernels numerically verified.


## Cell 14 — Per-Token Latency Benchmark

Measure per-token latency for the Duvar model using `torch.cuda.Event` for sub-millisecond precision.

In [14]:
def run_benchmark(model_obj, tokenizer, prompt_text: str, iterations: int = 10) -> float:
    """
    Measure per-token latency using torch.cuda.Event for sub-ms precision.
    Returns mean ms/token across all iterations.
    """
    enc = tokenizer(prompt_text, return_tensors="pt").to("cuda")

    # Warmup
    with torch.no_grad():
        _ = model_obj.generate(**enc, max_new_tokens=5)

    latencies = []
    for i in range(iterations):
        start = torch.cuda.Event(enable_timing=True)
        end   = torch.cuda.Event(enable_timing=True)
        start.record()
        with torch.no_grad():
            _ = model_obj.generate(**enc, max_new_tokens=128, do_sample=False)
        end.record()
        torch.cuda.synchronize()
        ms_per_token = start.elapsed_time(end) / 128
        latencies.append(ms_per_token)
        print(f"  Iteration {i+1:2d}: {ms_per_token:.2f} ms/token")

    return float(np.mean(latencies))


BENCH_PROMPT = (
    "How does the pointer-based graph architecture differ from "
    "standard sequential neural network execution?"
)

print("Benchmarking Duvar model...")
mean_ms = run_benchmark(model, tokenizer, BENCH_PROMPT, iterations=10)
print(f"\nResult: {mean_ms:.2f} ms/token (mean over 10 iterations)")
print("Note: dequant overhead is per-forward-pass; weight caching would reduce this.")

Benchmarking Duvar model...
  Iteration  1: 0.65 ms/token
  Iteration  2: 0.70 ms/token
  Iteration  3: 0.65 ms/token
  Iteration  4: 0.60 ms/token
  Iteration  5: 0.67 ms/token
  Iteration  6: 0.70 ms/token
  Iteration  7: 0.66 ms/token
  Iteration  8: 0.71 ms/token
  Iteration  9: 0.65 ms/token
  Iteration 10: 0.72 ms/token

Result: 0.67 ms/token (mean over 10 iterations)
Note: dequant overhead is per-forward-pass; weight caching would reduce this.
